# ViT Interpretability

This notebook inspects the selected Intel scene classifier checkpoint with three complementary views:

- input-gradient saliency maps
- ViT attention rollout maps
- nearest-neighbor inspection in ViT CLS embedding space

The selected checkpoint is `checkpoints/intel_vit_transfer_4_epoch_0060.pt`.

In [ ]:
from pathlib import Path
import sys
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Subset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset import IntelImageClassificationDataset
from src.inference.prediction import load_inference_bundle

torch.set_grad_enabled(False)

CHECKPOINT = PROJECT_ROOT / "checkpoints" / "intel_vit_transfer_4_epoch_0060.pt"
DATA_ROOT = PROJECT_ROOT / "data" / "intel"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

bundle = load_inference_bundle(CHECKPOINT, device=DEVICE)
model = bundle.model.eval()

dataset = IntelImageClassificationDataset(
    images_path=str(DATA_ROOT / "seg_test" / "seg_test"),
    transform=bundle.transform,
)

class_names = list(bundle.class_names)
print(f"device={bundle.device}")
print(f"samples={len(dataset)}")
print(class_names)

## Helpers

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(tensor):
    image = tensor.detach().cpu() * IMAGENET_STD + IMAGENET_MEAN
    return image.clamp(0, 1).permute(1, 2, 0).numpy()

def open_raw_image(index):
    return Image.open(dataset.image_paths[index]).convert("RGB")

def model_input(index):
    image = open_raw_image(index)
    tensor = bundle.transform(image).unsqueeze(0).to(bundle.device)
    label = int(dataset.labels[index])
    return image, tensor, label

def predict_tensor(tensor):
    with torch.inference_mode():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        confidence, pred = probs.max(dim=1)
    return logits, probs, int(pred.item()), float(confidence.item())

def display_image_from_tensor(tensor):
    return denormalize(tensor.detach().cpu())

def show_image_with_prediction(index, ax=None):
    image, tensor, label = model_input(index)
    _, _, pred, confidence = predict_tensor(tensor)
    display_image = display_image_from_tensor(tensor[0])
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(display_image)
    ax.axis("off")
    ax.set_title(
        f"true: {class_names[label]}\n"
        f"pred: {class_names[pred]} ({confidence:.3f})"
    )
    return pred, confidence

def pick_examples(per_class=1, only_wrong=False):
    selected = []
    for class_id, class_name in enumerate(class_names):
        candidates = [i for i, label in enumerate(dataset.labels) if int(label) == class_id]
        random.shuffle(candidates)
        for index in candidates:
            _, tensor, _ = model_input(index)
            _, _, pred, _ = predict_tensor(tensor)
            if only_wrong and pred == class_id:
                continue
            selected.append(index)
            if len([i for i in selected if int(dataset.labels[i]) == class_id]) >= per_class:
                break
    return selected

example_indices = pick_examples(per_class=1, only_wrong=False)
wrong_indices = pick_examples(per_class=1, only_wrong=True)
print("example indices:", example_indices)
print("wrong indices:", wrong_indices)

## Sample Predictions

Use these examples to orient the later saliency and attention maps. `wrong_indices` is useful for inspecting failure cases.

In [ ]:
indices_to_show = example_indices[:6]
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for ax, index in zip(axes.ravel(), indices_to_show):
    show_image_with_prediction(index, ax=ax)
plt.tight_layout()

## Saliency Maps

This section computes an input-gradient saliency map for the predicted class. Bright regions are pixels where small input changes most affect the selected class logit.

This is not Grad-CAM. For ViT, input saliency is simpler and avoids choosing an arbitrary convolutional target layer.

In [ ]:
def saliency_for_index(index, target_class=None):
    image, tensor, label = model_input(index)
    tensor = tensor.detach().clone().requires_grad_(True)
    model.zero_grad(set_to_none=True)

    with torch.enable_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        pred = int(probs.argmax(dim=1).item())
        selected_class = pred if target_class is None else int(target_class)
        score = logits[0, selected_class]
        score.backward()

    saliency = tensor.grad.detach().abs().max(dim=1).values[0]
    saliency = saliency - saliency.min()
    saliency = saliency / saliency.max().clamp_min(1e-8)
    return image, tensor.detach()[0], saliency.cpu(), label, pred, float(probs[0, pred].item())

def plot_saliency(index, target_class=None):
    image, input_tensor, saliency, label, pred, confidence = saliency_for_index(index, target_class)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    display_image = display_image_from_tensor(input_tensor)
    axes[0].imshow(display_image)
    axes[0].set_title(f"true: {class_names[label]} (224x224 model input)")
    axes[0].axis("off")
    axes[1].imshow(saliency, cmap="magma")
    axes[1].set_title("saliency")
    axes[1].axis("off")
    axes[2].imshow(display_image)
    axes[2].imshow(saliency, cmap="magma", alpha=0.45)
    axes[2].set_title(f"pred: {class_names[pred]} ({confidence:.3f})")
    axes[2].axis("off")
    plt.tight_layout()

plot_saliency(example_indices[0])

In [ ]:
for index in wrong_indices[:3]:
    plot_saliency(index)

## ViT Attention Rollout

Torchvision's ViT forward pass does not return attention weights. This helper replays each encoder block's self-attention with `need_weights=True`, averages heads, adds the residual identity, and rolls attention forward across layers.

The result is a class-token-to-patch attention map. It is a useful inspection tool, but it should not be treated as a causal explanation.

In [ ]:
def attention_rollout(tensor, discard_ratio=0.0):
    vit = model
    with torch.inference_mode():
        x = vit._process_input(tensor)
        batch_size = x.shape[0]
        cls_token = vit.class_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = vit.encoder.dropout(x)

        seq_len = x.shape[1]
        rollout = torch.eye(seq_len, device=x.device).unsqueeze(0).repeat(batch_size, 1, 1)

        for block in vit.encoder.layers:
            y = block.ln_1(x)
            _, attn = block.self_attention(
                y, y, y,
                need_weights=True,
                average_attn_weights=False,
            )
            attn = attn.mean(dim=1)

            if discard_ratio > 0:
                flat = attn[:, 1:, 1:].reshape(batch_size, -1)
                _, indices = flat.topk(int(flat.shape[1] * discard_ratio), largest=False)
                flat.scatter_(1, indices, 0)

            identity = torch.eye(seq_len, device=x.device).unsqueeze(0)
            attn = attn + identity
            attn = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            rollout = torch.bmm(attn, rollout)
            x = block(x)

        patch_attention = rollout[:, 0, 1:]
        grid_size = int(math.sqrt(patch_attention.shape[-1]))
        patch_attention = patch_attention.reshape(batch_size, 1, grid_size, grid_size)
        patch_attention = F.interpolate(
            patch_attention,
            size=tensor.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )[:, 0]
        patch_attention = patch_attention - patch_attention.amin(dim=(1, 2), keepdim=True)
        patch_attention = patch_attention / patch_attention.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)
    return patch_attention.cpu()

def plot_attention_rollout(index):
    image, tensor, label = model_input(index)
    _, _, pred, confidence = predict_tensor(tensor)
    attention = attention_rollout(tensor)[0]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    display_image = display_image_from_tensor(tensor[0])
    axes[0].imshow(display_image)
    axes[0].set_title(f"true: {class_names[label]} (224x224 model input)")
    axes[0].axis("off")
    axes[1].imshow(attention, cmap="viridis")
    axes[1].set_title("attention rollout")
    axes[1].axis("off")
    axes[2].imshow(display_image)
    axes[2].imshow(attention, cmap="viridis", alpha=0.45)
    axes[2].set_title(f"pred: {class_names[pred]} ({confidence:.3f})")
    axes[2].axis("off")
    plt.tight_layout()

plot_attention_rollout(example_indices[0])

In [ ]:
for index in wrong_indices[:3]:
    plot_attention_rollout(index)

## CLS Embedding Neighborhoods

This section embeds a balanced subset of validation images using the ViT CLS token, then retrieves nearest neighbors by cosine similarity.

Use it to inspect whether mistakes are close to visually similar examples, mislabeled-looking examples, or ambiguous class boundaries.

In [ ]:
def vit_cls_embedding(tensor):
    vit = model
    with torch.inference_mode():
        x = vit._process_input(tensor)
        batch_size = x.shape[0]
        cls_token = vit.class_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = vit.encoder(x)
        return x[:, 0]

def balanced_subset_indices(per_class=60):
    selected = []
    for class_id in range(len(class_names)):
        candidates = [i for i, label in enumerate(dataset.labels) if int(label) == class_id]
        random.shuffle(candidates)
        selected.extend(candidates[:per_class])
    return selected

def build_embedding_index(per_class=60, batch_size=64):
    indices = balanced_subset_indices(per_class=per_class)
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=0)
    embeddings = []
    labels = []
    for images, batch_labels in loader:
        images = images.to(bundle.device)
        embeddings.append(vit_cls_embedding(images).cpu())
        labels.extend([int(label) for label in batch_labels])
    embeddings = torch.cat(embeddings)
    embeddings = F.normalize(embeddings, dim=1)
    return indices, embeddings, labels

neighbor_indices, neighbor_embeddings, neighbor_labels = build_embedding_index(per_class=60)
print(neighbor_embeddings.shape)

In [ ]:
def nearest_neighbors(query_index, k=8):
    _, query_tensor, query_label = model_input(query_index)
    query_embedding = F.normalize(vit_cls_embedding(query_tensor).cpu(), dim=1)
    similarities = (neighbor_embeddings @ query_embedding[0]).numpy()
    order = np.argsort(-similarities)

    rows = []
    for position in order:
        candidate_index = neighbor_indices[int(position)]
        if candidate_index == query_index:
            continue
        rows.append((candidate_index, float(similarities[position])))
        if len(rows) >= k:
            break
    return rows

def plot_neighbors(query_index, k=8):
    neighbors = nearest_neighbors(query_index, k=k)
    total = k + 1
    cols = min(5, total)
    rows = math.ceil(total / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.4 * rows))
    axes = np.array(axes).reshape(-1)

    show_image_with_prediction(query_index, ax=axes[0])
    axes[0].set_title("QUERY\n" + axes[0].get_title())

    for ax, (index, similarity) in zip(axes[1:], neighbors):
        pred, confidence = show_image_with_prediction(index, ax=ax)
        ax.set_title(f"sim {similarity:.3f}\n" + ax.get_title())

    for ax in axes[total:]:
        ax.axis("off")
    plt.tight_layout()

plot_neighbors(example_indices[0])

In [ ]:
for index in wrong_indices[:3]:
    plot_neighbors(index)

## Suggested Follow-ups

- Save saliency and attention figures for the largest known failure pairs: `glacier`/`mountain` and `buildings`/`street`.
- Compare embedding neighborhoods for correct high-confidence, wrong high-confidence, and low-confidence samples.
- Add a small curated panel to `MODEL_CARD.md` if specific examples are cleared for inclusion.